# Course notebook — Transformers and Foundation Models

Understand the architecture, training, inference, and model families behind modern generative AI.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The meeting notes with the wrong owner

Asha reviews action items extracted from messy project transcripts. A teammate named twice is incorrectly assigned ownership for a different task. She needs a way to explain the failure to the project lead and compare a small local model with a hosted candidate without asking the team to trust the larger model by default.

The workplace scenario is fictional.


## Sequence models and attention


### Lecture 01 · Sequence Models Before Transformers

Understand n-grams, recurrent networks, LSTMs, encoder–decoder models, bottlenecks, and why long-range dependencies and serial computation were difficult.

#### The person named at the end

A decision begins on page one and ownership is assigned near the end. A sequence model that compresses history into a small state has little room for intervening discussion and the original thread.

Test short and long versions of the same transcript. If attribution worsens with distance, compare with a mechanism that can directly access earlier token representations. Hold the action item and speaker constant while changing the gap.

#### Read the mechanism

**n-grams**

N-gram models predict tokens from local history of n−1 prior tokens—simple, fast, and limited to short context. They remain baselines for compression and sanity checks.

A trigram model captures 'New York' but not dependencies spanning whole paragraphs.

**RNNs**

Recurrent neural networks process sequences step by step, maintaining hidden state across time. Serial computation limits parallel training and long-range credit assignment.

Character-level RNN language models learn spelling but struggle with paragraph-level coherence.

**LSTMs**

LSTMs add gating to RNNs to mitigate vanishing gradients and capture longer dependencies than plain RNNs. They dominated seq2seq before transformers but remain in some streaming pipelines.

LSTM encoders for time-series logs capture hourly patterns over days of context.

**seq2seq**

Sequence-to-sequence models map input sequences to output sequences via encoder–decoder architectures. They underpin translation, summarization, and tool-output generation patterns.

An encoder compresses ticket text; a decoder generates structured JSON fields.

**bottlenecks**

Information bottlenecks force compressive representations—fixed-size context vectors or limited bandwidth channels. They create trade-offs between memory and expressiveness.

Early seq2seq used a single context vector for entire sentences, losing detail on long inputs.

#### Worked example

**Situation:** A team must explain why decoding settings change model output and latency. They prototype next-token prediction with n-grams before adopting transformers.

**Baseline:** Trigram model over support macros—fails when incident description exceeds three-token context.

**Application:** Train n-gram on ticket corpus, identify failure at long-range dependency ("region" ... "failover"), contrast with RNN-style hidden state carry (simulated) showing bottleneck.

**Test cases:** (1) Normal: complete trigram match in template. (2) Boundary: context exactly at n-gram window edge. (3) Adversarial: repeated padding tokens dilute probability mass.

**Measurement:** Perplexity vs context length; latency per token for serial RNN simulation vs parallel n-gram lookup.

**Design question:** At what context length does the n-gram baseline break on the book scenario, and why?

**Predict before running.** Architectures evolve in response to information-flow and optimization bottlenecks. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 4.1: Sequence Models Before Transformers"""

CHAPTER = "4.1"
print("chapter hook:", CHAPTER)
from collections import Counter
text = "region east failover region west failover"
n = 3
grams = Counter(tuple(text.split()[i:i+n]) for i in range(len(text.split())-n+1))
context = ("region", "east")
candidates = [g[-1] for g in grams if g[:2] == context]
print({"context": context, "next_token_candidates": candidates})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Train an n-gram model and inspect where local context fails.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why did long-range dependencies motivate architectures beyond n-grams?
2. What symptom shows an RNN bottleneck without mentioning transformers?
3. What n-gram order baseline should precede neural sequence models?

<details><summary>Check your reasoning</summary>

Q1: n-grams cannot relate distant tokens; probability tables explode. Q2: Hidden state saturates—early tokens forgotten in long tickets. Q3: Fixed-order n-gram with same corpus and perplexity eval.

</details>


### Lecture 02 · Attention

Build attention from queries, keys, values, similarity scores, normalization, and weighted aggregation.

#### Attention can favor the wrong mention

The transcript mentions Jules twice in a budget discussion and once in the final ownership sentence. The extractor attaches the budget owner to the deliverable.

Attention creates input-dependent mixtures of representations. It does not guarantee the weighting reflects the role required by the task. Test with a transcript that repeats a distractor and check the observable owner assignment before using attention weights as an explanation.

#### Read the mechanism

**queries**

In attention, queries represent what information a position seeks from other positions. They are learned projections of hidden states, not user search queries.

Each decoder token issues a query vector to attend over encoder keys during translation.

**keys**

Keys are attention projections indexed for lookup—compatible queries receive high weights. Together with values they implement content-addressable memory over sequences.

A pronoun's query should match keys at its antecedent position for correct coreference routing.

**values**

Values carry the content aggregated by attention weights—what actually flows between positions. Weighted sums of values update each position's representation.

Attending to a verb's value brings predicate information into the subject's representation.

**scaled dot product**

Scaled dot-product attention computes softmax(QKᵀ/√d)V, scaling dot products to stable gradients. It is the core operation inside transformer blocks.

Without scaling, large dimensions push softmax into near one-hot distributions and vanishing gradients.

**attention masks**

Attention masks zero out disallowed positions—future tokens in decoding, padding, or cross-segment boundaries. Masks enforce causality and ignore irrelevant tokens.

Causal masks prevent a language model from peeking at answer tokens during training.

#### Worked example

**Situation:** Engineers need intuition for why certain tokens in a policy snippet receive more weight when summarizing an incident tied to that policy.

**Baseline:** Uniform averaging of token vectors—ignores relevance.

**Application:** Implement scaled dot-product attention: queries from summary slot, keys/values from policy tokens, visualize weight distribution over "outage", "SLA", "escalation."

**Test cases:** (1) Normal: query token aligns with one key. (2) Boundary: all keys orthogonal—uniform weights. (3) Adversarial: one key with huge norm dominates without scaling.

**Measurement:** Entropy of attention weights; summarization ROUGE vs uniform baseline on three snippets.

**Design question:** Why divide dot products by sqrt(d_k) before softmax in production-sized models?

**Predict before running.** Attention is content-dependent routing of information. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 4.2: Attention"""

import math
q = [1.0, 0.0]
keys = [[0.9, 0.1], [0.0, 1.0], [0.9, 0.1]]
def scaled_dot(q, k, scale):
    return sum(a*b for a, b in zip(q, k)) / scale
scale = math.sqrt(len(q))
scores = [scaled_dot(q, k, scale) for k in keys]
m = max(scores)
weights = [math.exp(s-m) for s in scores]
Z = sum(weights)
weights = [w/Z for w in weights]
print("weights:", [round(w, 3) for w in weights])


#### Interpret and apply

Implement scaled dot-product attention and visualize weights.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What does attention compute that a fixed convolution cannot?
2. How would an unscaled dot product distort weights when dimension grows?
3. What baseline aggregation ignores content-dependent routing?

<details><summary>Check your reasoning</summary>

Q1: Dynamic routing based on query-key compatibility. Q2: Large dimensions inflate dot products → sharp softmax → vanishing gradients for other keys. Q3: Mean pooling over token vectors.

</details>


**Section checkpoint.** Trace one attention query and identify which key contributes most to the weighted result.


## Transformer blocks and training


### Lecture 03 · The Transformer Block

Compose multi-head attention, feed-forward layers, residual paths, normalization, positional information, and masking.

#### Follow one representation through the block

Asha wants to know why replacing a name also changes the task category. The model passes contextualized representations from attention through other transformations before predicting.

Track tensor shapes and isolate one change in the input. Compare the resulting owner and category outputs. This connects the block-level mechanism to behavioral evidence without assuming any individual head is assigned a stable human-readable role.

#### Read the mechanism

**multi-head attention**

Multi-head attention runs several attention operations in parallel with separate projections, letting different heads capture diverse relations. Heads are often redundant but increase capacity.

One head may track syntax; another tracks coreference in the same layer.

**residual connections**

Residual connections add layer inputs to outputs, easing gradient flow through deep stacks. They let layers learn incremental refinements instead of full remappings.

Transformer blocks compute attention(x) + x rather than attention(x) alone.

**normalization**

Text normalization lowercases, strips diacritics, standardizes whitespace, and canonicalizes equivalents before indexing or tokenization. Over-normalization destroys discriminative identifiers.

Collapsing hyphens in SKUs merges distinct product codes; preserving case matters for camelCase APIs.

**MLP blocks**

MLP blocks apply position-wise feed-forward networks after attention, adding nonlinear capacity per token. They typically expand dimension 4× then project back.

FFN layers store factual associations in some interpretability studies of LMs.

**position**

Position information tells transformers token order since self-attention is permutation-invariant without it. Methods include sinusoidal, learned, and rotary (RoPE) encodings.

Rotary embeddings encode relative position in Q/K products for long-context models.

#### Worked example

**Situation:** The team assembles a minimal transformer block to predict the next token in incident summaries, ensuring tensor shapes flow correctly through attention and FFN.

**Baseline:** Single attention head without residuals—training unstable on small data.

**Application:** Stack multi-head attention (2 heads), residual connections, layer norm, and two-layer FFN; verify shape (batch, seq, dim) at each sub-layer; apply causal mask for autoregressive training.

**Test cases:** (1) Normal: seq_len=8, dim=16. (2) Boundary: seq_len=1 (degenerate attention). (3) Adversarial: mask bug allowing peek at future tokens.

**Measurement:** Training loss stability with/without residuals; shape assertion pass rate in unit tests.

**Design question:** Which component—residual path or normalization—would you remove first to demonstrate training failure?

**Predict before running.** Depth repeatedly mixes information and transforms representations. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 4.3: The Transformer Block"""

seq, dim, heads = 4, 8, 2
assert dim % heads == 0
head_dim = dim // heads
x = [[0.1 * (i+j) for j in range(dim)] for i in range(seq)]
def layer_norm(row):
    mu = sum(row) / len(row)
    var = sum((v-mu)**2 for v in row) / len(row)
    return [(v-mu)/(var+1e-5)**0.5 for v in row]
out = [layer_norm(row) for row in x]
print({"seq": seq, "dim": dim, "head_dim": head_dim, "row0_norm_mean": round(sum(out[0])/len(out[0]), 3)})


#### Interpret and apply

Assemble one transformer block and test tensor shapes.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why do residual connections help deep transformer stacks?
2. What bug does a causal mask prevent in autoregressive training?
3. What shallow baseline lacks multi-head mixing?

<details><summary>Check your reasoning</summary>

Q1: Gradients flow around sublayers; identity path stabilizes early training. Q2: Position t attends to t+1, leaking future tokens. Q3: Single-head attention without FFN or residuals.

</details>


### Lecture 04 · Training Foundation Models

Study autoregressive, masked, and sequence-to-sequence objectives; data mixtures; scaling; checkpoints; and mixture-of-experts.

#### Learn the task without seeing the answer key

The transcript collection contains meeting minutes and near-duplicate action-item exports. An evaluation set built from the same exports looks much cleaner than the actual meeting traffic.

Split by meeting series before creating windows, then deduplicate across splits. Keep evaluation examples from unseen teams and newer recordings. Otherwise a pretraining or fine-tuning comparison can reward memorization of workflow artifacts.

#### Read the mechanism

**pretraining objectives**

Pretraining objectives define self-supervised targets—causal LM, masked LM, denoising—that shape what models learn from raw text. Objective choice affects bidirectionality and use cases.

Causal LM suits generation; masked LM suits understanding tasks before fine-tuning.

**data mixtures**

Data mixtures blend corpora—web, code, books, dialog—at tuned ratios during pretraining. Mixture proportions strongly affect capabilities and biases.

Over-weighting code improves programming but may hurt conversational tone.

**scaling laws**

Scaling laws relate model size, data, and compute to predictable loss improvements—guiding budget allocation. They are approximate and domain-dependent.

Doubling parameters may yield diminishing returns if data quality does not scale similarly.

**checkpoints**

Checkpoints persist durable agent state so interrupted runs resume without repeating side effects.

After approval gate, checkpoint stores pending payment until human approves, then continues.

**mixture of experts**

Mixture-of-experts activates subsets of parameters per token, scaling capacity without proportional compute. Routing and load balancing add engineering complexity.

An MoE layer may route math tokens to specialized experts while sharing common language experts.

#### Worked example

**Situation:** Leadership asks for compute and data estimates to pretrain a tiny domain language model on internal policies without mistaking pretraining for a fact database.

**Baseline:** Assume memorizing all policies guarantees correct answers at inference.

**Application:** Estimate tokens in corpus, parameters for small GPT-style model, training steps given batch and context; distinguish pretraining objective (next-token) from downstream QA needs.

**Test cases:** (1) Normal: 10M tokens, 50M params. (2) Boundary: corpus dominated by duplicated templates. (3) Adversarial: contaminated eval documents inside pretrain mix.

**Measurement:** Tokens/param ratio, estimated GPU-hours, contamination check pass rate.

**Design question:** What evidence would show the model compressed statistical patterns rather than storing retrievable policy text verbatim?

**Predict before running.** Pretraining compresses statistical regularities into parameters; it does not create a fact database. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 4.4: Training Foundation Models"""

CHAPTER = "4.4"
print("chapter hook:", CHAPTER)
tokens, params, epochs, batch = 10_000_000, 50_000_000, 1, 32
steps = tokens // (batch * 512)
ratio = tokens / params
print({"train_steps_approx": steps, "tokens_per_param": round(ratio, 2)})
print("note: pretrain learns distributions, not a queryable policy DB")
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Estimate compute and data requirements for a tiny language model.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why does pretraining not create a reliable fact database?
2. How does template duplication distort scaling estimates?
3. What baseline uses retrieval instead of pretraining for policy facts?

<details><summary>Check your reasoning</summary>

Q1: Models interpolate and hallucinate; facts need grounding mechanisms. Q2: Effective tokens << raw tokens—inflated capacity estimates. Q3: RAG over authoritative policy index with same QA eval.

</details>


**Section checkpoint.** Draw the path through attention and the feed-forward block. Mark where training computes a loss.


## Decoding and model selection


### Lecture 05 · Inference and Sampling

Trace logits, softmax, temperature, top-k, top-p, streaming, batching, KV cache, prefix cache, and speculative decoding.

#### The selector needs repeatability

The team reruns the same transcript three times and sees two owners inferred. Asha can accept variation in notes phrasing, but an ownership field must stay grounded.

Tune output behavior by field and enforce constraints downstream. Measure owner consistency and correctness across repeated runs, rather than assuming a single low-temperature run establishes determinism or quality.

#### Read the mechanism

**logits**

Logits are raw pre-softmax scores over the vocabulary for the next token. Decoding policies—temperature, top-k—operate on logits before sampling.

Inspecting logits reveals whether the model hesitates between two equally likely tokens.

**sampling**

Sampling draws next tokens from the predicted distribution rather than always taking the argmax. It enables diverse outputs but introduces nondeterminism unless seeded.

Creative writing uses sampling; factual extraction often uses greedy or low-temperature decoding.

**temperature**

Temperature scales logits before softmax—lower sharpens the distribution (more deterministic), higher flattens it (more random). It is a primary creativity-versus-consistency knob.

Temperature 0.2 keeps support answers stable; 1.2 increases phrasing variety for marketing copy.

**KV cache**

The KV cache stores key and value tensors for prior tokens during autoregressive decoding, avoiding recomputation of the prefix. Memory grows linearly with context length.

Streaming chat reuses cached states for system prompt and prior turns, cutting latency after the first token.

**batching**

Batching groups requests to amortize GPU kernel overhead, improving throughput at possible latency cost. Continuous batching in servers interleaves sequences of different lengths.

Batch size 32 may double throughput versus batch 1 but increase p95 latency for short prompts.

#### Worked example

**Situation:** A team must explain why decoding settings change model output and latency when serving incident summaries from a local model.

**Baseline:** Greedy decoding (argmax) only—deterministic but often repetitive.

**Application:** Build sampling playground: logits → temperature-scaled softmax → top-k and top-p filters; simulate KV cache hit on repeated prefix tokens; compare tokens/sec with and without cache.

**Test cases:** (1) Normal: temperature=0.7, top_p=0.9. (2) Boundary: temperature→0 approaches greedy. (3) Adversarial: top_k=1 still stochastic if temperature high.

**Measurement:** Output diversity (distinct n-grams), latency per token, cache memory vs prefix length.

**Design question:** When does KV caching stop helping because the prefix changes every request?

**Predict before running.** Generation is repeated conditional prediction shaped by decoding and system context. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 4.5: Inference and Sampling"""

import random
logits = [2.0, 1.0, 0.5, 0.1]
def sample_temp(logits, temp=1.0):
    scaled = [l/temp for l in logits]
    m = max(scaled)
    ex = [math.exp(l-m) for l in scaled]
    s = sum(ex)
    probs = [e/s for e in ex]
    r = random.random()
    c = 0
    for i, p in enumerate(probs):
        c += p
        if r <= c:
            return i, probs
    return len(probs)-1, probs
import math
idx, probs = sample_temp(logits, temp=0.8)
print({"sampled_index": idx, "probs": [round(p, 3) for p in probs]})


#### Interpret and apply

Build a sampling playground and compare decoding strategies.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. How does temperature change the sampling distribution?
2. Why does KV cache reduce latency in autoregressive decoding?
3. What decoding baseline removes all randomness?

<details><summary>Check your reasoning</summary>

Q1: Lower temp sharpens distribution toward argmax; higher flattens it. Q2: Prefix keys/values reused instead of recomputed each step. Q3: Greedy argmax at temperature 0.

</details>


### Lecture 06 · Model Families and Selection

Compare base, instruction, reasoning, code, embedding, reranking, reward, safety, speech, vision, and diffusion models.

#### Choose a model with the reviewers

A hosted model wins on long transcripts. The local model matches it on ordinary meetings and can keep sensitive transcripts on the approved machine.

Route by evaluated needs and permission, then include review time and escalation rates in the model decision. A larger model has a defensible role if measured benefit exceeds latency and access costs for the actual workload.

#### Read the mechanism

**instruction tuning**

Instruction tuning fine-tunes models on prompt–response pairs covering diverse tasks, improving zero-shot instruction following. It shapes helpfulness and format compliance.

After instruction tuning, models follow 'respond in JSON' without task-specific fine-tuning.

**reasoning models**

Reasoning models allocate extra inference compute—long chains, self-checks—for math, code, and planning tasks. They trade latency and cost for accuracy on hard problems.

A reasoning model may emit scratchpad steps before the final answer on a budget word problem.

**multimodal models**

Multimodal models ingest text, images, audio, or video in shared architectures for joint understanding or generation. Modality alignment and tokenization differ per input type.

A vision-language model answers questions about chart images in earnings reports.

**open weights**

Open-weights models publish parameters for local deployment, fine-tuning, and inspection—versus API-only access. They shift control, compliance, and operational burden to your team.

Self-hosting Llama enables air-gapped inference but requires GPU ops and security patching.

**model routing**

Model routing directs requests to appropriate models by task, risk, cost, or latency policy.

Regex on ticket category routes billing to fine-tuned small model, general to large.

#### Worked example

**Situation:** Platform team must pick among base, instruction-tuned, code, embedding, and reranker models for the policy assistant—vendor marketing overwhelms requirements.

**Baseline:** Choose the largest model name on the leaderboard regardless of task.

**Application:** Define task-specific dataset (QA, citation, routing), benchmark candidates on accuracy/latency/cost slices, document when to route simple queries to small instruct model and hard ones to reasoning model.

**Test cases:** (1) Normal: straightforward policy lookup. (2) Boundary: query needing reranker after retrieval. (3) Adversarial: benchmark prompt leaked in training data inflates scores.

**Measurement:** Task success by model tier, $/1k requests, rollback time when swapping models.

**Design question:** Which requirement would force a dedicated reranker instead of a larger generative model?

**Predict before running.** Select models as replaceable components against requirements, not by reputation. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 4.6: Model Families and Selection"""

tasks = {"lookup": 0.95, "cite": 0.88, "route": 0.91}
models = {"small-instruct": 0.01, "large-reason": 0.08}
def route(task, risk):
    if risk == "low" and tasks[task] > 0.9:
        return "small-instruct"
    return "large-reason"
for risk in ("low", "high"):
    print({"risk": risk, "model": route("lookup", risk), "cost_per_1k": models[route("lookup", risk)]})


#### Interpret and apply

Benchmark candidate models on a task-specific dataset.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why select models against requirements rather than reputation?
2. When does an embedding model replace a generative model in the stack?
3. What baseline routes everything to one flagship model?

<details><summary>Check your reasoning</summary>

Q1: Leaderboard tasks rarely match enterprise slices or governance constraints. Q2: Pure retrieval/ranking steps need vectors or cross-encoders, not generation. Q3: Single largest LLM for all endpoints.

</details>


**Section checkpoint.** Compare decoding distributions at two temperatures. Pick a model selection metric for repeatability.


## Course workshop

Keep logits fixed, vary temperature, and compare the probability of the leading token. Explain why selection still needs a task-level evaluation.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Scaled dot-product attention and temperature sampling without dependencies."""
from math import exp, sqrt
from random import Random


def softmax(values, temperature=1.0):
    scaled = [v / temperature for v in values]; peak = max(scaled)
    weights = [exp(v - peak) for v in scaled]; total = sum(weights)
    return [w / total for w in weights]


def attention(query, keys, values):
    scores = [sum(q * k for q, k in zip(query, key)) / sqrt(len(query)) for key in keys]
    weights = softmax(scores)
    output = [sum(weight * value[i] for weight, value in zip(weights, values)) for i in range(len(values[0]))]
    return weights, output


weights, output = attention([1, 0], [[1, 0], [0, 1]], [[10, 0], [0, 10]])
print("attention_weights=", [round(x, 3) for x in weights], "output=", [round(x, 3) for x in output])
for temperature in (0.3, 1.0, 2.0):
    probabilities = softmax([3.0, 2.0, 1.0], temperature)
    print(f"temperature={temperature}: {[round(x, 3) for x in probabilities]}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
cold = softmax([2.0, 1.0, 0.0], temperature=0.5)
warm = softmax([2.0, 1.0, 0.0], temperature=2.0)
assert abs(sum(cold) - 1.0) < 1e-9
assert cold[0] > warm[0]
print('Workshop checks passed')


## Course project

Implement a tiny transformer and create a vendor-neutral model selection report.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
